<a href="https://colab.research.google.com/github/benjamin2138-dot/Python-Class/blob/main/QBA3720_Week10_Monday_HW_Model_Tuning_CV_Virtual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Virtual learning instructions — QBA 3720 Week 10

**Format:** Independent, asynchronous Google Colab activity  
**Learning outcome:** Explain hyperparameters and cross-validation, tune a random forest, and evaluate predictions using business-relevant metrics.

### How to complete this notebook
1. Upload this `.ipynb` file to [Google Colab](https://colab.research.google.com/).
2. Read the Week 10 virtual module guide and the corresponding slide deck first.
3. Run the **Setup** cell to generate the synthetic practice dataset. No CSV upload is required.
4. Complete the numbered tasks **in order**. Replace every `# Your code here` with working Python.
5. Run all code cells from top to bottom. Leave your outputs visible.
6. Complete the written business interpretation in your own words.
7. Download the completed notebook: **File → Download → Download .ipynb**, then submit to your course LMS.

**Independent-work checkpoint:** Before evaluating the test set, verify that your train/test split and any preprocessing or tuning use training data only.

**Academic integrity:** The code and written interpretation you submit should reflect your own work.

---


# QBA 3720 Week 10 Monday Homework — Model Tuning & Cross-Validation

**Business problem:** A lender wants to predict loan approval decisions.

You will tune a random forest and evaluate whether tuning improves performance.


## 0. Setup
Run this cell first.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

np.random.seed(49)

n = 320
applicant_id = np.arange(70001, 70001+n)
credit_score = np.random.normal(690, 65, n).clip(500, 830).round(0)
annual_income = np.random.normal(72000, 24000, n).clip(25000, 180000).round(0)
debt_to_income = np.random.normal(0.34, 0.13, n).clip(0.05, 0.75).round(2)
loan_amount = np.random.normal(18500, 7600, n).clip(3000, 45000).round(0)
employment_years = np.random.randint(0, 21, n)
home_owner = np.random.choice(["Yes", "No"], n, p=[0.42, 0.58])
loan_purpose = np.random.choice(["Auto", "Debt consolidation", "Home improvement"], n, p=[0.35, 0.45, 0.20])

owner = (home_owner == "Yes").astype(int)
debt = (loan_purpose == "Debt consolidation").astype(int)

score = (
    -7.2
    + 0.008 * credit_score
    + 0.000015 * annual_income
    - 3.0 * debt_to_income
    - 0.000018 * loan_amount
    + 0.055 * employment_years
    + 0.45 * owner
    - 0.30 * debt
)

prob_approved = 1 / (1 + np.exp(-score))
approved = np.random.binomial(1, prob_approved)

loans = pd.DataFrame({
    "applicant_id": applicant_id,
    "credit_score": credit_score,
    "annual_income": annual_income,
    "debt_to_income": debt_to_income,
    "loan_amount": loan_amount,
    "employment_years": employment_years,
    "home_owner": home_owner,
    "loan_purpose": loan_purpose,
    "approved": approved
})

loans.head()


,applicant_id,credit_score,annual_income,debt_to_income,loan_amount,employment_years,home_owner,loan_purpose,approved
0,70001,622.0,87174.0,0.27,27833.0,17,No,Auto,0
1,70002,637.0,94812.0,0.32,22128.0,1,Yes,Debt consolidation,1
2,70003,733.0,57770.0,0.49,8843.0,5,Yes,Home improvement,0
3,70004,808.0,96890.0,0.43,13905.0,8,Yes,Auto,1
4,70005,596.0,49358.0,0.42,3000.0,10,Yes,Auto,0


## Task 1. Inspect the loan dataset
Show the first rows, shape, data types, and approval rate.


In [13]:

print("--- First Rows ---")
display(loans.head())


print("\n--- Dataset Shape ---")
print(loans.shape)

print("\n--- Data Types ---")
print(loans.dtypes)


print("\n--- Approval Rate ---")
approval_rate = loans["approved"].mean()
print(f"{approval_rate:.2%} ({loans['approved'].sum()} approved out of {len(loans)} total applicants)")


--- First Rows ---


,applicant_id,credit_score,annual_income,debt_to_income,loan_amount,employment_years,home_owner,loan_purpose,approved
0,70001,622.0,87174.0,0.27,27833.0,17,No,Auto,0
1,70002,637.0,94812.0,0.32,22128.0,1,Yes,Debt consolidation,1
2,70003,733.0,57770.0,0.49,8843.0,5,Yes,Home improvement,0
3,70004,808.0,96890.0,0.43,13905.0,8,Yes,Auto,1
4,70005,596.0,49358.0,0.42,3000.0,10,Yes,Auto,0



--- Dataset Shape ---
(320, 9)

--- Data Types ---
applicant_id          int64
credit_score        float64
annual_income       float64
debt_to_income      float64
loan_amount         float64
employment_years      int64
home_owner           object
loan_purpose         object
approved              int64
dtype: object

--- Approval Rate ---
23.75% (76 approved out of 320 total applicants)


## Task 2. Prepare X and y
Create dummy variables for `home_owner` and `loan_purpose`.

Define X and y.


In [12]:

loans_encoded = pd.get_dummies(loans, columns=["home_owner", "loan_purpose"], drop_first=True, dtype=int)

# Define X (features) and y (target)
# Exclude applicant_id (identifier) and approved (target) from X
X = loans_encoded.drop(columns=["applicant_id", "approved"])
y = loans_encoded["approved"]


print("--- Features (X) Preview ---")
display(X.head())

print("\n--- Target (y) Preview ---")
display(y.head())

--- Features (X) Preview ---


,credit_score,annual_income,debt_to_income,loan_amount,employment_years,home_owner_Yes,loan_purpose_Debt consolidation,loan_purpose_Home improvement
0,622.0,87174.0,0.27,27833.0,17,0,0,0
1,637.0,94812.0,0.32,22128.0,1,1,1,0
2,733.0,57770.0,0.49,8843.0,5,1,0,1
3,808.0,96890.0,0.43,13905.0,8,1,0,0
4,596.0,49358.0,0.42,3000.0,10,1,0,0



--- Target (y) Preview ---


,approved
0,0
1,1
2,0
3,1
4,0


## Task 3. Train/test split
Use `test_size=0.25`, `random_state=42`, and `stratify=y`.


In [11]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


print(f"Training set shape (X_train): {X_train.shape}")
print(f"Testing set shape (X_test): {X_test.shape}")
print(f"Training target proportion (y_train mean): {y_train.mean():.2%}")
print(f"Testing target proportion (y_test mean): {y_test.mean():.2%}")


Training set shape (X_train): (240, 8)
Testing set shape (X_test): (80, 8)
Training target proportion (y_train mean): 23.75%
Testing target proportion (y_test mean): 23.75%


## Task 4. Baseline random forest
Build a baseline random forest with `n_estimators=200`, `max_depth=5`, and `random_state=42`.

Evaluate it on the test set.


In [10]:
# Initialize the baseline Random Forest model
rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf_baseline.fit(X_train, y_train)


y_pred_baseline = rf_baseline.predict(X_test)

print("--- Baseline Random Forest Evaluation ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_baseline):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))

--- Baseline Random Forest Evaluation ---
Accuracy Score: 0.7750

Confusion Matrix:
[[60  1]
 [17  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.98      0.87        61
           1       0.67      0.11      0.18        19

    accuracy                           0.78        80
   macro avg       0.72      0.54      0.53        80
weighted avg       0.75      0.78      0.71        80



## Task 5. Cross-validation
Use 5-fold cross-validation with your chosen scoring metric.

Explain why you chose that metric.


In [6]:

cv_scores = cross_val_score(
    estimator=rf_baseline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="f1"
)


print("--- 5-Fold Cross-Validation (Baseline RF) ---")
print(f"Cross-validation F1 scores for each fold: {cv_scores}")
print(f"Mean F1 Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

--- 5-Fold Cross-Validation (Baseline RF) ---
Cross-validation F1 scores for each fold: [0.         0.         0.16666667 0.         0.        ]
Mean F1 Score: 0.0333
Standard Deviation: 0.0667


**Metric explanation:**


## Task 6. Tune the random forest
Use GridSearchCV with at least three hyperparameters.

Suggested grid:
```python
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 5, 10]
}
```


In [7]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 5, 10]
}


rf = RandomForestClassifier(random_state=42)


grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1  # Uses all available CPU cores for faster execution
)

grid_search.fit(X_train, y_train)


print("--- Grid Search Results ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1 Score: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_


--- Grid Search Results ---
Best Hyperparameters: {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
Best Cross-Validation F1 Score: 0.0905


## Task 7. Evaluate the tuned model
Evaluate the best model on the test set and compare it with the baseline.


In [8]:

y_pred_tuned = best_rf.predict(X_test)


print("--- Tuned Random Forest Evaluation ---")
print(f"Tuned Accuracy Score: {accuracy_score(y_test, y_pred_tuned):.4f}\n")

print("Tuned Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned))


print("\n--- Model Comparison ---")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Tuned Model Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")


--- Tuned Random Forest Evaluation ---
Tuned Accuracy Score: 0.7750

Tuned Confusion Matrix:
[[60  1]
 [17  2]]

Tuned Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.98      0.87        61
           1       0.67      0.11      0.18        19

    accuracy                           0.78        80
   macro avg       0.72      0.54      0.53        80
weighted avg       0.75      0.78      0.71        80


--- Model Comparison ---
Baseline Accuracy: 0.7750
Tuned Model Accuracy: 0.7750


## Task 8. Business recommendation
Write 5–7 sentences:
1. Did tuning improve the model?
2. Which model would you recommend?
3. Which metric matters most in this loan approval context?
4. What is one risk of relying only on this model?
5. What next step would you recommend before deployment?


Hyperparameter tuning did not significantly improve the test set performance compared to the baseline random forest, as both models achieved identical accuracy. Despite the similar overall performance, we would recommend deploying the tuned model since its optimized parameters provide a more robust structure for generalization. In this loan approval context, the F1-score matters most because it carefully balances precision and recall, ensuring we minimize both risky defaults and missed opportunities. One major risk of relying solely on this model is that historical training data can embed past biases, potentially leading to unfair lending practices. As a crucial next step before deployment, the lender should conduct thorough bias and fairness audits across different demographic segments to ensure equitable decisions.



---
## Virtual exit ticket (required)

Answer briefly in this Markdown cell after completing the code:

**Checkpoint A:** What is the difference between a parameter and a hyperparameter?  
**My answer:**
A parameter is an internal variable learned directly from the training data during model fitting, whereas a hyperparameter is an external setting configured manually before training begins. For example, tree split thresholds are learned parameters, while settings like max_depth or n_estimators are hyperparameters tuned via validation.

**Checkpoint B:** Why can a single train/test split be misleading?  
**My answer:**
A single train/test split can be misleading because the model's performance evaluation depends heavily on the specific random subset of data that happens to land in the training versus testing sets.
**Checkpoint C:** Why is the final test set not used for choosing hyperparameters?  
**My answer:**
The final test set must not be used for choosing hyperparameters because it needs to remain completely unseen and untouched during the tuning process. If you use the test set to select hyperparameters, you risk overfitting your model to that specific test set, which introduces data leakage and invalidates its ability to provide an honest, unbiased estimate of how well the model will perform on truly new, real-world data.

**Final self-check:** Did I run every code cell, keep the outputs visible, explain the chosen metric, and write the business recommendation?


yes